In [0]:
CREATE TABLE IF NOT EXISTS de_learning_workspace.silver.products
(
    product_id INT,
    product_name STRING,
    category STRING,
    brand STRING,
    price DECIMAL(10,2),
    stock_available INT,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    _load_timestamp TIMESTAMP,
    _source_system STRING,

    _silver_processed_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
%python
from pyspark.sql.functions import (
    col,
    trim,
    when,
    current_timestamp
)

bronze_df = spark.table("de_learning_workspace.bronze.products")

silver_df = (
    bronze_df

    # Remove duplicate products
    .dropDuplicates(["product_id"])

    # Clean product name
    .withColumn(
        "product_name",
        when(
            (col("product_name").isNull()) |
            (trim(col("product_name")) == ""),
            "Unknown"
        ).otherwise(
            trim(col("product_name"))
        )
    )

    # Clean category
    .withColumn(
        "category",
        when(
            (col("category").isNull()) |
            (trim(col("category")) == ""),
            "Unknown"
        ).otherwise(
            trim(col("category"))
        )
    )

    # Clean brand
    .withColumn(
        "brand",
        when(
            (col("brand").isNull()) |
            (trim(col("brand")) == ""),
            "Unknown"
        ).otherwise(
            trim(col("brand"))
        )
    )

    # Processing metadata
    .withColumn(
        "_silver_processed_timestamp",
        current_timestamp()
    )
)

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("de_learning_workspace.silver.products")
)